# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">7주차 · 셀 대신 스크립트 하나로 추천 결과 파일을 만든다</mark>

지난 여섯 주 동안 추천은 **셀을 하나씩 눌러서** 만들었습니다.

실제 서비스에서는 사람이 매번 셀을 누르지 않습니다. **실행 한 번**으로 모든 고객의 추천이 파일로 나와야 합니다.

오늘은 6주차의 최종 추천(행렬분해 + 위험도 상한)을 그대로 두고, **만드는 방법만** 바꿉니다.

1. **학습과 서빙을 나눈다** — 무거운 학습은 한 번만, 추천은 그 결과를 꺼내 쓰기만
2. **일괄 추천** — 300명의 추천을 한 번에 미리 계산해 표 하나로
3. **결과 파일 규격** — 다음 주 앱이 읽을 파일의 모양을 약속하고 검사
4. **실행 스크립트 하나** — 이 모든 일을 `python recommend_batch.py` 한 줄로

---

### 오늘의 구성

| 파트 | 어디서 | 하는 일 |
|---|---|---|
| 개념 | 슬라이드 | 학습과 서빙 · 일괄 추천 · 결과 파일 규격 · 재현성 |
| 실습 | **이 노트북** | 함수로 묶고, 파일로 저장하고, 스크립트로 한 번에 실행한다 |
| 마무리 | 슬라이드 | 오늘의 용어 · 점수판 · 다음 주 |

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기 · 이 노트북을 여는 법</mark>

1. https://colab.research.google.com/github/welovecherry/recsys/blob/main/notebooks/07_batch_pipeline.ipynb
2. 구글 계정으로 로그인합니다.
3. **경고창이 뜨면 `Run anyway` 를 누릅니다.**
4. **아래 셀의 ▶ 버튼을 누릅니다.** 실습 자료를 받아옵니다. 10초쯤 걸립니다.
5. 그다음부터는 위에서 아래로 셀을 하나씩 실행하면 됩니다.

> ⚠ **고친 내용을 남기려면** 메뉴에서 `파일 → 드라이브에 사본 저장` 을 눌러 주세요.

---

**이 노트북이 쓰는 것**

| 무엇 | 쓰임 |
|---|---|
| `data/items.csv` · `users.csv` · `interactions.csv` | 종목 100개 · 투자자 300명 · 거래 기록 8,086건 |
| `recsys.py` | 데이터 읽기 · 시간순 분할 · 채점 · 사용자-아이템 행렬 |
| `recommend_batch.py` | **오늘의 산출물** — 실행 한 번으로 결과 파일을 만드는 스크립트 |

**이 노트북이 만드는 것**

| 무엇 | 어디에 |
|---|---|
| 300명 × 추천 10개 결과 파일 | `output/recommendations.csv` |

⚠ **지난주 노트북을 먼저 돌릴 필요가 없습니다.** 이 노트북 하나로 끝납니다.

---

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">먼저 이 셀부터 실행하세요  ▶</mark>

In [1]:
# 이 셀에서 하는 일 — Colab 에서 열었으면 실습 자료를 내려받는다
# 왜 하나 — Colab 은 열 때마다 빈 컴퓨터라 데이터·recsys.py 가 없다. 내 컴퓨터면 그냥 넘어간다

import os          # 폴더를 만들고 옮겨 다니는 도구
import sys         # 지금 파이썬이 어떤 환경인지 알려 주는 도구
import subprocess  # 터미널 명령을 파이썬에서 대신 실행해 주는 도구

if "google.colab" in sys.modules:                    # Colab 이면 이 안이 실행된다
    if os.path.exists("/content/recsys"):            # 전에 받아 둔 것이 있으면 최신으로
        subprocess.run(["git", "-C", "/content/recsys", "pull", "-q", "--ff-only"])
    else:                                            # 처음이면 통째로 내려받는다
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/welovecherry/recsys.git", "/content/recsys"])
    os.chdir("/content/recsys/notebooks")            # 노트북 폴더 안으로 이동
    print("준비 끝 —", os.getcwd(), "· 아래 셀부터 차례로 실행하세요.")
else:
    print("내 컴퓨터에서 실행 중입니다 —", os.getcwd(), "· 따로 받아올 것이 없습니다.")

내 컴퓨터에서 실행 중입니다 — /Users/hong/workspaces/org_physical-spark/course-recsys/notebooks · 따로 받아올 것이 없습니다.


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">오늘 나오는 말 여섯 개</mark>

1. **학습 (training)** — 거래 기록으로 모델을 만드는 일
   - 오늘은 `TruncatedSVD` 로 사용자 요인 · 아이템 요인을 만드는 일입니다
   - 계산이 무겁고, 하루에 한 번처럼 **가끔** 합니다

2. **서빙 (serving)** — 학습한 결과로 고객에게 추천을 **꺼내 주는** 일
   - 계산이 가볍고, 고객이 앱을 열 때마다 **자주** 합니다
   - 학습과 서빙을 나누면 서빙할 때 다시 학습할 필요가 없습니다

3. **일괄 추천 (batch recommendation)** — 모든 고객의 추천을 **한꺼번에 미리** 계산해 두는 방식
   - 앱은 미리 만든 결과를 읽기만 합니다

4. **결과 파일 규격 (output schema)** — 결과 파일의 **열 이름 · 순서 · 개수**를 정해 둔 약속
   - 만드는 쪽(오늘 스크립트)과 읽는 쪽(다음 주 앱)이 같은 약속을 지켜야 파일이 맞물립니다

5. **실행 스크립트 (script)** — 노트북 셀 대신 **파일 하나**에 담아 한 번에 실행하는 프로그램
   - 오늘의 산출물 `recommend_batch.py` 입니다

6. **재현성 (reproducibility)** — 같은 입력으로 다시 돌리면 **같은 결과**가 나오는 성질
   - `random_state=42` 처럼 무작위 요소를 고정해서 지킵니다

---

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">준비 — 데이터 읽기와 시간순 분할</mark>

**여기서 하는 일** — 데이터를 읽고 3주차부터 쓰던 **시간순 분할**로 나눕니다.

- 채점은 지난주와 똑같이 합니다. 그래야 오늘 점수를 6주차 **0.3899** 와 비교할 수 있습니다.

In [2]:
# 이 셀에서 하는 일 — 도구를 불러오고, 데이터를 읽고, 시간순으로 나눈다
# 왜 하나 — 지난주와 같은 기준으로 채점해야 0.3899 와 비교할 수 있다

import sys                                  # 파이썬이 파일을 찾는 경로를 다루는 도구
sys.path.insert(0, ".")                     # 지금 폴더에서 recsys.py 를 찾게 한다
sys.path.insert(0, "notebooks")             # 한 단계 안쪽 폴더(notebooks)도 찾게 한다

import numpy as np                          # 숫자 묶음을 빠르게 다루는 도구. np 라고 부른다
import pandas as pd                         # 표를 다루는 도구. pd 라고 부른다
import recsys                               # 이 수업용 도구 모음 — 읽기 · 분할 · 채점이 들어 있다
from sklearn.decomposition import TruncatedSVD   # 4주차에 쓴 행렬분해 도구

items, users, interactions = recsys.load()                  # 종목 · 투자자 · 거래 기록 세 표
train, test, 기준시점 = recsys.split_by_time(interactions)   # 3주차부터 쓰던 시간순 분할
성향 = users.set_index("user_id")["persona"].to_dict()        # 딕셔너리 {투자자 번호: 투자성향}
위험도 = items.set_index("item_id")["risk_level"].to_dict()   # 딕셔너리 {종목 번호: 1~5}

print(f"학습 구간 {len(train):,}건 · 채점 구간 {len(test):,}건 · 거래 기록 전체 {len(interactions):,}건")

학습 구간 7,095건 · 채점 구간 991건 · 거래 기록 전체 8,086건


**결과 읽는 법**

- 거래 기록 8,086건을 자른 결과입니다.
  - **학습 구간 7,095건**으로 모델을 만들고, **채점 구간 991건**으로 맞혔는지 확인합니다.
  - 오늘은 **거래 기록 전체 8,086건**도 씁니다. 7.3 에서 서비스용 모델을 만들 때입니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">7.1 준비 — 6주차 최종 추천을 함수 두 개로</mark>

**7.1 에서 하는 일**
- 6주차의 최종 추천(행렬분해 + 위험도 상한)을 **함수 두 개**로 묶습니다.
  - `학습하기(기록)` — 학습. 서빙에 필요한 것을 딕셔너리 하나(**학습 결과 묶음**)에 담아 돌려줍니다.
  - `추천하기(묶음, 사람)` — 서빙. 그 묶음만 보고 한 사람에게 추천 10개를 줍니다.

**왜 둘로 나누나**
- 학습은 무겁고, 추천은 가볍습니다. 7.2 에서 실제로 재 봅니다.
- 나눠 두면 **학습은 한 번만** 하고, 추천은 몇 번이든 묶음에서 꺼내 쓸 수 있습니다.

In [3]:
# 이 셀에서 하는 일 — 학습 함수를 만든다. 서빙에 필요한 것을 딕셔너리 하나에 담는다
# 왜 하나 — 학습 결과를 한 덩어리로 들고 다녀야 서빙이 학습과 따로 움직일 수 있다

상한 = {"안정형": 3, "성장형": 4, "공격형": 5}          # 6주차 적합 기준 — 투자성향별 최대 위험도

def 학습하기(기록):                                      # def = 함수를 만든다. 거래 기록을 받는다
    """거래 기록으로 TruncatedSVD 를 학습하고, 서빙에 필요한 것을 딕셔너리 하나에 담는다."""
    상호작용_행렬 = recsys.interaction_matrix(기록, items)    # 사용자-아이템 행렬 — 담았으면 1
    모델 = TruncatedSVD(n_components=8, random_state=42)     # 잠재 요인 8개 · 무작위 고정
    return {"사용자_요인": 모델.fit_transform(상호작용_행렬),  # 학습 — 사용자 요인
            "아이템_요인": 모델.components_,                   # 아이템 요인
            "사용자_목록": list(상호작용_행렬.index),           # 행 이름 (투자자 번호)
            "아이템_목록": list(상호작용_행렬.columns),         # 열 이름 (종목 번호)
            "인기순위": list(기록["item_id"].value_counts().index),             # 기록 없는 사람용
            "이미_담은것": 기록.groupby("user_id")["item_id"].apply(set).to_dict()}  # 뺄 종목

묶음_채점용 = 학습하기(train)                             # 채점용은 학습 구간만으로 학습
print("학습 결과 묶음에 든 것 :", list(묶음_채점용))

학습 결과 묶음에 든 것 : ['사용자_요인', '아이템_요인', '사용자_목록', '아이템_목록', '인기순위', '이미_담은것']


**결과 읽는 법**

- 학습 결과 묶음은 **이름표 여섯 개**가 붙은 딕셔너리입니다.
  - 사용자 요인 · 아이템 요인 — 6주차까지 쓰던 행렬분해 결과
  - 사용자 목록 · 아이템 목록 — 행렬의 행 이름과 열 이름
  - 인기순위 · 이미 담은 것 — 추천할 때 필요한 나머지 재료
- 이 묶음만 있으면 거래 기록을 다시 읽지 않고도 추천할 수 있습니다.

In [4]:
# 이 셀에서 하는 일 — 점수 순서에서 위험도 상한을 걸고 10개를 고르는 함수를 만든다
# 왜 하나 — 6주차의 「위험도 상한 → 이미 담은 것 빼기」를 서빙 함수가 쓰게 따로 둔다

def 상한_걸고_10개(묶음, 사람, 순서, 점수표):
    """순서에서 상한을 넘는 종목과 이미 담은 종목을 빼고 위에서 10개."""
    통과 = []                                              # 상한을 통과한 종목
    for 번호 in 순서:                                      # 점수가 높은 종목부터
        if 위험도[번호] <= 상한[성향[사람]]:                # <= 작거나 같다. 상한 이하만
            통과.append(번호)                              # 목록 끝에 붙인다
    결과 = []                                              # (종목 번호, 점수) 를 담을 목록
    for 번호 in recsys.take(통과, 묶음["이미_담은것"].get(사람, set())):   # take = 이미 담은 것 빼고 10개
        결과.append((번호, round(점수표.get(번호, 0.0), 4)))  # 점수는 소수 넷째 자리까지
    return 결과

In [5]:
# 이 셀에서 하는 일 — 서빙 함수를 만든다. 묶음만 보고 한 사람에게 추천 10개를 준다
# 왜 하나 — 점수 순서를 만들고, 바로 위 함수로 거르고 10개를 고른다

def 추천하기(묶음, 사람):                                 # 학습 결과 묶음과 투자자 번호를 받는다
    """위험도 상한을 지킨 추천 10개를 (종목 번호, 점수) 목록으로 준다."""
    순서, 점수표 = 묶음["인기순위"], {}                    # 기록이 없는 사람은 인기 순서 · 점수 없음
    if 사람 in 묶음["사용자_목록"]:                        # 기록이 있는 사람은 행렬분해 점수 순서
        점수 = 묶음["사용자_요인"][묶음["사용자_목록"].index(사람)] @ 묶음["아이템_요인"]   # @ = 내적
        순서 = []                                          # 점수가 높은 종목부터 담을 목록
        for 자리 in np.argsort(-점수):                     # argsort(-점수) = 큰 것부터 자리 번호
            순서.append(묶음["아이템_목록"][자리])          # 자리 번호 → 종목 번호
            점수표[묶음["아이템_목록"][자리]] = float(점수[자리])   # 나중에 파일에 적을 점수
    return 상한_걸고_10개(묶음, 사람, 순서, 점수표)        # 아래 함수가 거르고 10개를 고른다

print("U0001 에게 줄 추천 앞 3개 :", 추천하기(묶음_채점용, "U0001")[:3])

U0001 에게 줄 추천 앞 3개 : [('I002', 0.603), ('I043', 0.4803), ('I033', 0.4056)]


**결과 읽는 법**

- U0001 에게 줄 추천 앞 3개를 **(종목 번호, 점수)** 짝으로 보여 줍니다.
  - 첫 번째 짝의 I002 는 TIGER 미국배당다우존스이고, 점수는 4주차 행렬분해 점수입니다.
- 6주차와 같은 방식이므로, 이 목록에는 U0001(안정형)의 상한 3 을 넘는 종목이 없습니다.

In [6]:
# 이 셀에서 하는 일 — 두 함수로 만든 추천을 3주차와 같은 기준으로 채점한다
# 왜 하나 — 함수로 묶어도 6주차와 같은 추천인지 0.3899 로 확인한다

def 채점용_추천(사람, 이미):                             # recsys 채점 함수가 요구하는 모양 (사람, 이미)
    """채점 함수에 넘길 수 있게 종목 번호만 돌려준다."""
    번호들 = []                                            # 종목 번호만 담을 목록
    for 번호, 점수 in 추천하기(묶음_채점용, 사람):          # (번호, 점수) 짝을 하나씩
        번호들.append(번호)                                # 번호만 모은다
    return 번호들

채점_점수, 인원 = recsys.recall_at_k(채점용_추천, train, test)  # 3주차부터 쓰던 채점 함수
print(f"함수로 묶은 추천 — Recall@10 = {채점_점수:.4f}   (채점 대상 {인원}명 · 6주차 0.3899)")

함수로 묶은 추천 — Recall@10 = 0.3899   (채점 대상 266명 · 6주차 0.3899)


**결과 읽는 법**

- 함수 두 개로 묶은 추천의 Recall@10 이 **0.3899** 입니다.
  - 6주차 위험도 상한 추천과 같은 값입니다. **추천은 그대로이고, 묶는 방법만 바뀌었다**는 확인입니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">7.2 학습과 서빙 — 무거운 일은 한 번만</mark>

**7.2 에서 하는 일** — 학습 한 번과 추천 한 번이 각각 얼마나 걸리는지 **시간을 재서** 비교합니다.

**1. 무엇을 재나**
- 학습 — `학습하기(train)` 를 한 번
- 서빙 — `추천하기` 를 300명에게 한 번씩

**2. 주의할 점**
- 시간은 컴퓨터마다, 실행할 때마다 조금씩 다릅니다. **정확한 값보다 몇 배 차이인지**를 봅니다.
- `time.perf_counter()` 는 지금 시각을 아주 잘게 재는 시계입니다. 끝난 시각에서 시작 시각을 빼면 걸린 시간입니다.

In [7]:
# 이 셀에서 하는 일 — 학습 한 번과 서빙 300번에 걸린 시간을 잰다
# 왜 하나 — 학습과 서빙을 왜 나누는지 숫자로 확인한다

import time                                               # 시간을 재는 도구

시작 = time.perf_counter()                                # 시작 시각
학습하기(train)                                           # 학습 한 번
학습_시간 = time.perf_counter() - 시작                     # 끝난 시각 - 시작 시각

시작 = time.perf_counter()
for 사람 in users["user_id"]:                             # 300명에게 한 번씩
    추천하기(묶음_채점용, 사람)                             # 서빙 — 이미 만든 묶음에서 꺼내기만
서빙_시간 = (time.perf_counter() - 시작) / len(users)       # 한 사람당 평균

print(f"학습 한 번 {학습_시간 * 1000:.1f}ms · 한 사람 추천 {서빙_시간 * 1000:.3f}ms · 약 {학습_시간 / 서빙_시간:,.0f}배")

학습 한 번 51.6ms · 한 사람 추천 0.039ms · 약 1,314배


**결과 읽는 법**

- 첫 숫자는 **학습 한 번**, 둘째 숫자는 **한 사람에게 추천 한 번**에 걸린 시간입니다. 단위 ms 는 1,000분의 1초입니다.
- 마지막 숫자는 학습 한 번이 추천 한 번보다 **몇 배 오래 걸리는지**입니다.
  - 실행할 때마다 조금씩 다르지만, 학습이 **수백 배 이상** 무겁다는 것은 같습니다.
- 우리 데이터는 종목 100개 · 투자자 300명뿐이라 둘 다 빠릅니다. 데이터가 커지면 학습 시간은 훨씬 길어집니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">7.3 일괄 추천 — 300명 × 10개를 미리 계산</mark>

**7.3 에서 하는 일**
- **서비스용 모델**을 거래 기록 **전체**로 다시 학습합니다.
- 그 모델로 300명 모두의 추천 10개를 **한꺼번에** 계산해 표 하나(3,000줄)로 만듭니다.

**1. 왜 서비스용은 전체 기간으로 학습하나**
- 채점할 때는 마지막 한 달을 **숨겨야** 했습니다. 실력을 정직하게 재려고요(3주차).
- 서비스용 추천은 **내일 고객에게 보여 줄** 것입니다. 가장 최근 기록까지 모두 써야 합니다.

**2. 표의 한 줄**
- 한 줄 = 한 사람의 추천 한 개입니다. 300명 × 10개 = **3,000줄**입니다.

In [8]:
# 이 셀에서 하는 일 — 서비스용 모델을 거래 기록 전체로 학습한다
# 왜 하나 — 내일 고객에게 보여 줄 추천은 가장 최근 기록까지 써야 한다

묶음_서비스용 = 학습하기(interactions)                     # 전체 기간 8,086건으로 학습
print(f"서비스용 묶음 — 행렬에 있는 투자자 {len(묶음_서비스용['사용자_목록'])}명 (채점용은 {len(묶음_채점용['사용자_목록'])}명)")
print("U0001 채점용 앞 3개 :", 추천하기(묶음_채점용, "U0001")[:3])
print("U0001 서비스용 앞 3개 :", 추천하기(묶음_서비스용, "U0001")[:3])

서비스용 묶음 — 행렬에 있는 투자자 300명 (채점용은 285명)
U0001 채점용 앞 3개 : [('I002', 0.603), ('I043', 0.4803), ('I033', 0.4056)]
U0001 서비스용 앞 3개 : [('I002', 0.6211), ('I043', 0.4705), ('I014', 0.3974)]


**결과 읽는 법**

- 첫 줄 — 서비스용 묶음에는 투자자 **300명** 모두가 들어 있습니다.
  - 채점용(학습 구간)에는 285명뿐이었습니다. 마지막 한 달에 처음 온 신규 15명이 이제 들어왔습니다.
- 아래 두 줄 — 같은 U0001 이라도 **어느 기록으로 학습했느냐**에 따라 추천과 점수가 조금 달라집니다.

In [9]:
# 이 셀에서 하는 일 — 서비스용 묶음으로 300명 모두의 추천을 표 하나에 담는다 (일괄 추천)
# 왜 하나 — 앱이 고객마다 계산하지 않고, 미리 만든 표를 읽기만 하게 하려는 것이다

종목이름 = items.set_index("item_id")["name"].to_dict()     # 딕셔너리 {종목 번호: 이름}
줄들 = []                                                  # 표의 한 줄씩을 담을 목록
for 사람 in users["user_id"]:                              # 300명을 한 명씩
    순위 = 1                                               # 1등부터 센다
    for 번호, 점수 in 추천하기(묶음_서비스용, 사람):          # 그 사람의 추천 10개
        줄들.append({"user_id": 사람, "rank": 순위, "item_id": 번호, "item_name": 종목이름[번호],
                     "risk_level": int(위험도[번호]), "persona": 성향[사람], "score": 점수})
        순위 = 순위 + 1                                    # 다음 등수로
추천표 = pd.DataFrame(줄들)                                # 목록 → 표
print(f"추천표 {추천표.shape}  ← 3,000줄 × 열 7개")
print(추천표.head(3).to_string(index=False))              # 앞 3줄만 보기

추천표 (3000, 7)  ← 3,000줄 × 열 7개
user_id  rank item_id                           item_name  risk_level persona  score
  U0001     1    I002                      TIGER 미국배당다우존스           2     안정형 0.6211
  U0001     2    I043 Vanguard FTSE Developed Markets ETF           3     안정형 0.4705
  U0001     3    I014                                  KT           2     안정형 0.3974


**결과 읽는 법**

- 첫 줄 `(3000, 7)` 은 추천표의 크기입니다. 300명 × 10개 = **3,000줄**, 열은 **7개**입니다.
- 아래 표는 앞 3줄, 곧 U0001 의 추천 1\~3등입니다.
  - 한 줄에 투자자 번호 · 순위 · 종목 번호 · 종목 이름 · 위험도 · 투자성향 · 점수가 있습니다.

**🗣 수업에서 함께 보기**

채점할 때는 **학습 구간만**으로 학습했는데, 서비스용 파일은 **거래 기록 전체**로 다시 학습했습니다.

3주차에서 「미래 기록으로 배우면 데이터 누수」라고 배웠는데, 이건 괜찮을까요? 그리고 두 모델의 추천은 얼마나 다를까요?

- 힌트 — 서비스용 추천은 **언제** 고객에게 보여 주는 것인지 생각해 보세요. 7.3 의 U0001 두 줄도 다시 보세요.
- 수업에서 함께 이야기합니다. 답은 노트북 맨 아래 **「답」**에도 있습니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">7.4 결과 파일 규격 — 다음 주 앱과의 약속</mark>

**7.4 에서 하는 일** — 추천표를 **정해 둔 규격**대로 CSV 파일 하나에 저장합니다.

**1. 규격 (이 수업에서 정한 약속)**

| 열 | 뜻 | 예 |
|---|---|---|
| `user_id` | 투자자 번호 | U0001 |
| `rank` | 추천 순위 1 \~ 10 | 1 |
| `item_id` | 종목 번호 | I002 |
| `item_name` | 종목 이름 | TIGER 미국배당다우존스 |
| `risk_level` | 위험도 1 \~ 5 | 2 |
| `persona` | 투자성향 | 안정형 |
| `score` | 행렬분해 점수 (기록이 없으면 0) | 0.6211 |

- 열은 **이 이름, 이 순서**로 7개입니다. 투자자마다 **정확히 10줄**입니다.
- 파일 자리는 `output/recommendations.csv` 입니다.

**2. 왜 규격을 정하나**
- 다음 주 앱은 이 파일을 읽기만 합니다. 열 이름 하나만 달라도 앱이 멈춥니다.
- 그래서 **만드는 쪽과 읽는 쪽이 같은 약속**을 미리 정해 둡니다.

**3. CSV 를 고른 이유**
- 엑셀이나 메모장으로도 열어 볼 수 있습니다.
- `encoding="utf-8-sig"` 로 저장하면 엑셀에서 한글이 깨지지 않습니다.

In [10]:
# 이 셀에서 하는 일 — 추천표를 규격 순서대로 CSV 파일 하나에 저장한다
# 왜 하나 — 다음 주 앱이 읽을 파일을 정해 둔 약속대로 만든다

import os                                                 # 폴더를 다루는 도구

규격_열 = ["user_id", "rank", "item_id", "item_name", "risk_level", "persona", "score"]   # 이 이름, 이 순서
추천표 = 추천표[규격_열]                                   # 열 순서를 규격에 맞춘다
os.makedirs("output", exist_ok=True)                       # output 폴더가 없으면 만든다
추천표.to_csv("output/recommendations.csv", index=False, encoding="utf-8-sig")   # index=False = 행 번호는 빼고
print(f"저장 — output/recommendations.csv · {len(추천표):,}줄 · {os.path.getsize('output/recommendations.csv'):,} 바이트")

저장 — output/recommendations.csv · 3,000줄 · 158,179 바이트


**결과 읽는 법**

- `output` 폴더에 `recommendations.csv` 가 생겼습니다.
  - 3,000줄은 7.3 의 추천표 그대로입니다.
  - 바이트는 파일 크기입니다. 1,000 바이트가 약 1KB 입니다.
- Colab 이라면 왼쪽 📁 아이콘을 누르면 `notebooks/output` 안에서 이 파일을 볼 수 있습니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">7.5 규격 검사 — 파일이 약속을 지켰나</mark>

**7.5 에서 하는 일** — 저장한 파일을 **다시 읽어서** 규격을 지켰는지 자동으로 검사합니다.

**검사하는 것 다섯 가지**
1. 열 이름과 순서가 규격과 같은가
2. 300명 모두 추천이 있는가
3. 사람마다 정확히 10줄인가
4. 투자성향의 상한을 넘는 종목이 없는가 (6주차 적합 기준)
5. 이미 담은 종목을 추천하지 않았는가

- 어긴 것이 있으면 목록으로 돌려주고, **빈 목록이면 통과**입니다.
- 사람이 3,000줄을 눈으로 확인할 수 없으니, 검사도 코드가 합니다.

In [11]:
# 이 셀에서 하는 일 — 결과 파일이 규격을 지켰는지 다섯 가지를 검사하는 함수를 만든다
# 왜 하나 — 3,000줄을 사람이 확인할 수 없다. 약속을 어긴 파일이 앱으로 넘어가지 않게 막는다

def 규격_검사(표, 묶음):                                  # 결과 표와 그 표를 만든 학습 결과 묶음
    """규격을 어긴 것을 목록으로 돌려준다. 빈 목록이면 통과."""
    문제 = []                                              # 어긴 것을 담을 목록
    if list(표.columns) != 규격_열:                         # 1. 열 이름 · 순서
        문제.append("열 이름이나 순서가 규격과 다릅니다")
    if 표["user_id"].nunique() != len(users):               # 2. nunique = 서로 다른 값의 개수
        문제.append("추천이 없는 사람이 있습니다")
    if (표.groupby("user_id").size() != 10).any():          # 3. size = 사람마다 줄 수 · any = 하나라도
        문제.append("추천이 10개가 아닌 사람이 있습니다")
    for _, 줄 in 표.iterrows():                             # iterrows = 표를 한 줄씩
        if 줄["risk_level"] > 상한[줄["persona"]]:           # 4. 상한을 넘으면
            문제.append(f"{줄['user_id']} 에게 상한을 넘는 {줄['item_id']}")
        if 줄["item_id"] in 묶음["이미_담은것"].get(줄["user_id"], set()):   # 5. 이미 담았으면
            문제.append(f"{줄['user_id']} 에게 이미 담은 {줄['item_id']}")
    return 문제

In [12]:
# 이 셀에서 하는 일 — 저장한 파일을 다시 읽어서 규격 검사를 한다
# 왜 하나 — 메모리의 표가 아니라 실제로 저장된 파일이 약속을 지켰는지 본다

읽은_표 = pd.read_csv("output/recommendations.csv", encoding="utf-8-sig")   # 저장한 파일 다시 읽기
문제 = 규격_검사(읽은_표, 묶음_서비스용)                    # 다섯 가지 검사
print(f"읽은 파일 {읽은_표.shape} · 문제 {len(문제)}개 →", "통과" if not 문제 else 문제[:3])

읽은 파일 (3000, 7) · 문제 0개 → 통과


**결과 읽는 법**

- 저장한 파일을 다시 읽으니 크기가 `(3000, 7)` 로 저장할 때와 같습니다.
- **문제 0개 → 통과** 입니다.
  - 열 이름 · 300명 · 사람마다 10줄 · 위험도 상한 · 이미 담은 종목, 다섯 가지를 모두 지켰습니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">7.6 실행 스크립트 — 셀 대신 파일 하나</mark>

**7.6 에서 하는 일** — 7.1 \~ 7.5 를 **파일 하나**로 묶은 `recommend_batch.py` 를 실행합니다.

**1. 스크립트가 하는 일**
1. 거래 기록 전체로 학습한다 — 7.1 의 `학습하기`
2. 300명의 추천 10개를 표 하나로 만든다 — 7.3
3. 규격 검사를 하고, 통과하면 `output/recommendations.csv` 로 저장한다 — 7.4 · 7.5

**2. 왜 노트북이 아니라 스크립트인가**
- 노트북은 사람이 셀을 위에서부터 **눌러야** 돌아갑니다.
- 스크립트는 `python recommend_batch.py` **한 줄**이면 끝까지 돌아갑니다. 매일 밤 정해진 시각에 컴퓨터가 알아서 실행하게 할 수도 있습니다.
- 이것이 계획서가 말한 **「사람 손을 뺀다」**입니다.

**3. 아래 셀에서 쓰는 도구**
- `subprocess.run` — 터미널 명령을 파이썬에서 대신 실행합니다. 맨 위 Colab 셋업 셀에서도 썼습니다.

In [13]:
# 이 셀에서 하는 일 — 스크립트 파일의 맨 앞 설명을 읽어 본다
# 왜 하나 — 실행하기 전에 이 파일이 무엇을 하는지 먼저 확인한다

with open("recommend_batch.py", encoding="utf-8") as 파일:   # with = 다 읽으면 파일을 저절로 닫는다
    줄들 = 파일.read().split("\n")                          # 파일 전체를 줄 단위로 나눈다
print(f"recommend_batch.py · {len(줄들)}줄. 맨 앞 설명 :")
for 줄 in 줄들[1:8]:                                       # 2번째 줄부터 7줄만
    print("  ", 줄)

recommend_batch.py · 106줄. 맨 앞 설명 :
   
       python recommend_batch.py
   
   하는 일 (6주차까지 노트북에서 셀을 하나씩 눌러 하던 일을 그대로 묶었다)
     1. 거래 기록 전체로 모델을 학습한다            — 학습(training)
     2. 300명 모두에게 위험도 상한을 지킨 추천 10개   — 일괄 추천(batch recommendation)
     3. 정해 둔 규격대로 CSV 파일 하나에 저장한다      — 결과 파일 규격


**결과 읽는 법**

- 첫 줄은 스크립트 파일의 길이(줄 수)입니다.
- 아래 일곱 줄은 파일 맨 앞의 설명입니다.
  - 실행 방법 `python recommend_batch.py` 와, 이 파일이 하는 일 네 가지(학습 · 일괄 추천 · 결과 파일 · 규격 검사)가 적혀 있습니다.

In [14]:
# 이 셀에서 하는 일 — 스크립트를 실행한다. 노트북 셀을 하나도 누르지 않은 것과 같다
# 왜 하나 — 실행 한 번으로 결과 파일까지 나오는지 확인한다

import subprocess                                          # 터미널 명령을 파이썬에서 대신 실행하는 도구

실행 = subprocess.run([sys.executable, "recommend_batch.py"], capture_output=True, text=True)   # sys.executable = 지금 쓰는 파이썬
print(실행.stdout.strip() or 실행.stderr.strip())          # stdout = 스크립트가 화면에 찍은 글자

완료 — output/recommendations.csv · 3,000줄 · 300명 · 규격 검사 통과


**결과 읽는 법**

- 스크립트가 끝나면서 찍은 한 줄입니다.
  - 파일 자리 · **3,000줄** · **300명** · **규격 검사 통과** 가 모두 적혀 있습니다.
- 노트북 셀을 하나도 누르지 않고, **실행 한 번**으로 학습부터 저장까지 끝났습니다.

In [15]:
# 이 셀에서 하는 일 — 스크립트가 만든 파일이 노트북에서 만든 추천표와 같은지 비교한다
# 왜 하나 — 셀을 파일 하나로 옮기면서 무언가 빠지거나 바뀌지 않았는지 확인한다

스크립트_표 = pd.read_csv("output/recommendations.csv", encoding="utf-8-sig")   # 스크립트가 만든 파일
같은가 = 스크립트_표.equals(읽은_표)                         # equals = 모든 칸이 같으면 True
print(f"스크립트가 만든 파일 {스크립트_표.shape} · 노트북에서 만든 파일과 같은가 : {같은가}")

스크립트가 만든 파일 (3000, 7) · 노트북에서 만든 파일과 같은가 : True


**결과 읽는 법**

- `True` 는 스크립트가 만든 파일과 노트북(7.4)에서 만든 파일이 **모든 칸까지 같다**는 뜻입니다.
- 셀 여러 개를 파일 하나로 옮겨도 **추천은 하나도 바뀌지 않았습니다.**

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">7.7 재현성 — 두 번 돌려도 같은 파일</mark>

**7.7 에서 하는 일** — 스크립트를 **한 번 더** 실행하고, 두 파일이 바이트 하나까지 같은지 비교합니다.

**왜 확인하나**
- 매일 밤 자동으로 돌리는 스크립트가 같은 기록으로 **다른 결과**를 낸다면, 어느 쪽이 맞는지 알 수 없습니다.
- 행렬분해에는 무작위 요소가 있습니다. `random_state=42` 로 고정해 두었기 때문에 같은 결과가 나와야 합니다.

**아래 셀에서 쓰는 도구**
- `shutil.copy` — 파일을 복사합니다.
- `filecmp.cmp(..., shallow=False)` — 두 파일을 **내용까지** 비교합니다. 같으면 True.

In [16]:
# 이 셀에서 하는 일 — 스크립트를 한 번 더 돌리고, 두 결과 파일이 똑같은지 비교한다
# 왜 하나 — 자동으로 매일 돌릴 스크립트는 같은 입력에 같은 결과를 내야 믿을 수 있다

import shutil, filecmp                                     # 파일 복사 도구 · 파일 비교 도구

shutil.copy("output/recommendations.csv", "output/first_run.csv")       # 첫 결과를 옆에 복사해 둔다
subprocess.run([sys.executable, "recommend_batch.py"], capture_output=True)   # 한 번 더 실행
같은_파일 = filecmp.cmp("output/recommendations.csv", "output/first_run.csv", shallow=False)   # 내용까지 비교
print(f"두 번 실행한 결과 파일이 바이트 하나까지 같은가 : {같은_파일}")

두 번 실행한 결과 파일이 바이트 하나까지 같은가 : True


**결과 읽는 법**

- `True` 는 두 번 실행한 결과 파일이 **바이트 하나까지 같다**는 뜻입니다.
- 같은 거래 기록이면 언제 돌려도 같은 추천이 나옵니다. 이것이 **재현성**입니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">7.8 오늘의 마무리</mark>

1. 6주차 최종 추천을 **학습하기 · 추천하기** 함수 두 개로 묶었습니다. 점수는 **0.3899** 그대로입니다.
2. 학습 한 번이 추천 한 번보다 **수백 배 이상** 무겁다는 것을 재 보고, 학습과 서빙을 나눴습니다.
3. 서비스용 모델을 거래 기록 **전체**로 학습해 300명 × 10개를 **일괄 추천**했습니다.
4. 열 7개짜리 **결과 파일 규격**을 정하고, 다섯 가지 **규격 검사**를 통과했습니다.
5. 이 모든 일을 **실행 스크립트 하나**(`recommend_batch.py`)로 묶었고, 두 번 돌려도 같은 파일이 나왔습니다(**재현성**).

**오늘 기억할 한 줄** — **추천은 그대로이고, 만드는 데 사람 손이 빠졌습니다.**

다음 주에는 이 결과 파일을 읽어서 **고객이 보는 앱 화면**을 만듭니다.

In [17]:
# 이 셀에서 하는 일 — 오늘 점수를 점수판에 남긴다
# 왜 하나 — 점수가 그대로인 주차도 적는다. 무엇이 바뀌었는지 말할 수 있으면 된다

recsys.record(7, "실행 스크립트로 자동화", 채점_점수,
              note="6주차 추천을 학습 · 일괄 추천 · 결과 파일 · 규격 검사로 묶었다. 점수는 그대로, 사람 손이 빠졌다")

recsys.leaderboard(upto=7)   # 오늘까지 쌓인 점수판 (뒤 주차는 빼고 본다)

레벨 7 · 실행 스크립트로 자동화 · Recall@10 = 0.3899


,level,name,recall_at_10,note
0,1,모두에게 같은 인기 순위,0.2163,알고리즘 없음. 인기 상위 10개를 모두에게 같게 추천했다
1,2,취향이 비슷한 세그먼트끼리,0.2684,거래 기록으로 주력 섹터와 평균 위험도를 뽑아 세그먼트로 나눴다
2,3,세그먼트별 목록 — 2주차와 같음,0.2684,추천 방식은 2주차와 같다. 무작위로 나누면 0.3150 이 나오는데 그것은 미래를...
3,4,행렬분해 SVD · 요인 8개,0.3918,규칙을 사람이 정하지 않았다. 사람 285 × 종목 100 표를 요인 8개로 쪼갰다
4,5,2단계 추천 · 후보 생성 + 순위 매기기,0.3382,2주차 세그먼트로 후보 40개를 추리고 그 40개만 4주차 모델로 정렬했다
5,6,투자 성향별 위험도 상한,0.3899,4주차 추천에서 성향 기준을 넘는 종목을 뺐다. 안 맞는 추천 205개 → 0개
6,7,실행 스크립트로 자동화,0.3899,6주차 추천을 학습 · 일괄 추천 · 결과 파일 · 규격 검사로 묶었다. 점수는 그...


**결과 읽는 법**

- 첫 줄은 오늘 점수를 점수판에 남겼다는 확인입니다. 7주차는 **0.3899** 로 6주차와 같습니다.
- 아래 표는 1주차부터 7주차까지의 점수판입니다.
  - `note` 끝의 `...` 은 글이 길어서 화면에서 잘린 것입니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📝 혼자 해 보기</mark>

수업이 끝난 뒤 혼자 풀어 보는 문제입니다. 문제마다 **어느 절의 결과를 보면 되는지** 제목에 적어 두었습니다.

답은 바로 아래 **「답」**에 있습니다. 먼저 스스로 생각해 본 뒤에 읽어 보세요.

**📝 혼자 해 보기 1 · 7.5 결과**

7.5 의 규격 검사를 **하지 않고** 파일을 다음 주 앱에 넘긴다고 해 봅시다.

다섯 가지 검사 가운데 하나라도 어긴 파일이 넘어가면, 고객 화면에서는 각각 어떤 일이 생길까요?

- 힌트 — 7.4 의 「왜 규격을 정하나」와 6주차의 적합성원칙을 떠올려 보세요.
- 답은 노트북 맨 아래 **「답」**에 있습니다.

**📝 혼자 해 보기 2 · 7.7 결과**

결과 파일에 **「파일을 만든 날짜와 시각」** 열을 하나 더 넣으면 좋겠다는 의견이 나왔습니다.

그렇게 하면 7.7 의 재현성 검사는 어떻게 될까요? 시각을 남기고 싶다면 어디에 남기는 게 좋을까요?

- 힌트 — 7.7 은 두 파일을 **바이트 하나까지** 비교합니다. 두 번 실행한 시각은 같을까요?
- 답은 노트북 맨 아래 **「답」**에 있습니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">답</mark>

먼저 스스로 답을 생각해 본 뒤에 읽어 보세요. 답에 나오는 숫자는 모두 이 노트북과 같은 코드로 실제로 계산한 값입니다.

**함께 보기 답 — 괜찮습니다. 서비스용 추천을 보여 주는 시점에는 그 기록이 모두 과거이기 때문입니다**

- 데이터 누수는 **채점할 때** 생기는 문제입니다.
  - 채점은 「모르는 미래를 얼마나 맞히나」를 재는 일이라, 맞힐 기간을 학습에 넣으면 실력이 부풀려집니다(3주차).
- 서비스용 추천은 **내일 고객에게 보여 줄** 추천입니다.
  - 그 시점에는 마지막 한 달도 이미 지난 과거입니다. 가장 최근 기록까지 모두 쓰는 것이 맞습니다.
  - 그래서 실무에서는 **채점용 모델로 실력을 재고, 서비스용 모델은 전체 기간으로 다시 학습**합니다.
- 두 모델의 추천은 꽤 비슷하지만 같지는 않습니다.
  - 300명의 추천 10개를 비교하면 평균 **7.45개**가 같고, 10개가 모두 같은 사람은 **11명**입니다.
  - 채점용에 없던 **신규 15명**도 서비스용에서는 자기 기록으로 추천을 받습니다.
- **한 줄로 정리하면** — 실력은 미래를 숨기고 재고, 서비스는 아는 것을 모두 써서 만듭니다.

**혼자 해 보기 1 답 — 앱이 멈추거나, 빈칸이 생기거나, 고객에게 맞지 않는 종목이 보입니다**

- **열 이름 · 순서가 다르면** — 앱이 `item_name` 같은 열을 찾지 못해 화면이 멈춥니다.
- **추천이 없는 사람이 있으면** — 그 고객의 화면에 추천 칸이 비어 나옵니다.
- **10줄이 아닌 사람이 있으면** — 화면의 추천 개수가 사람마다 달라지거나 칸이 어긋납니다.
- **상한을 넘는 종목이 있으면** — 안정형 고객에게 위험도 4 \~ 5 종목이 그대로 보입니다.
  - 6주차에서 본 적합성원칙에 어긋나는 추천이 고객에게 나가는 것입니다.
- **이미 담은 종목이 있으면** — 고객이 이미 가진 종목을 다시 추천해서 추천이 쓸모없어집니다.
- **한 줄로 정리하면** — 규격 검사는 잘못된 파일이 고객 화면까지 가지 않게 막는 마지막 확인입니다.

**혼자 해 보기 2 답 — 실행할 때마다 시각이 달라서 재현성 검사가 늘 실패합니다. 시각은 결과 파일 밖에 남깁니다**

- 두 번 실행한 시각은 다를 수밖에 없습니다.
  - 그래서 추천이 똑같아도 시각 열 때문에 두 파일이 달라지고, 7.7 의 비교는 `False` 가 됩니다.
- 그러면 「추천이 달라졌나」를 이 검사로 확인할 수 없게 됩니다.
- 시각을 남기고 싶다면 결과 파일 **밖**에 남깁니다.
  - 예: 파일 이름에 날짜를 넣거나(`recommendations_2026-11-10.csv`), 실행 기록을 적는 파일을 따로 둡니다.
- **한 줄로 정리하면** — 재현성을 확인하는 파일에는 실행할 때마다 바뀌는 값을 넣지 않습니다.